# gpt1 — Train a GPT from scratch on Colab

This notebook runs the full pipeline for the [gpt1](https://github.com/Razamindset/gpt1) project:

1. Mount Google Drive (all data / tokenizer / checkpoints / logs / plots persist there — a Colab disconnect won't lose your progress)
2. Clone the repo and install requirements
3. Download a training corpus (pick a size preset)
4. Train the BPE tokenizer
5. Train the GPT model, with live loss-curve plots and automatic checkpointing/resuming
6. Generate text from the trained model

**Before you start:** in Colab, go to `Runtime > Change runtime type` and pick a **GPU** (T4 is fine).

## 1. Mount Drive & get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

REPO_URL = "https://github.com/Razamindset/gpt1.git"
REPO_DIR = "/content/gpt1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Choose your project settings

Everything below is saved under `/content/drive/MyDrive/<PROJECT_NAME>/` on your Drive:
`data/`, `checkpoints/` (tokenizer + model checkpoints), `logs/`, `plots/`.

Re-running this notebook later with the same `PROJECT_NAME` will pick up right where you left off — dataset, tokenizer and training all resume automatically.

In [ ]:
PROJECT_NAME = "gpt1"        # change this to start a separate experiment
DATASET_PRESET = "small"     # "tiny" (~1MB), "small" (~3-4MB), "medium" (~8-10MB)
NUM_MERGES = 4000            # BPE vocabulary size (roughly)
EPOCHS = 20
BATCH_SIZE = 64


## 3. Download the dataset

Only re-downloads if you haven't already — safe to re-run.

In [ ]:
!python download_dataset.py --project {PROJECT_NAME} --preset {DATASET_PRESET}


## 4. Train the tokenizer

In [ ]:
import os
tokenizer_path = f"/content/drive/MyDrive/{PROJECT_NAME}/checkpoints/tokenizer.json"

if os.path.exists(tokenizer_path):
    print(f"Tokenizer already exists at {tokenizer_path}, skipping. Delete it to retrain.")
else:
    !python train_tokenizer.py --project {PROJECT_NAME} --num-merges {NUM_MERGES}


## 5. Train the model

This automatically resumes from `last_model.pt` if you already have one for this project
(e.g. after a Colab disconnect) — just re-run this cell.

In [ ]:
!python train.py --project {PROJECT_NAME} --epochs {EPOCHS} --batch-size {BATCH_SIZE}


## 6. View the training curves

In [ ]:
from IPython.display import Image, display

plot_path = f"/content/drive/MyDrive/{PROJECT_NAME}/plots/loss_curve.png"
display(Image(filename=plot_path))


## 7. Generate text

`--checkpoint` can be `best` (lowest validation loss), `last` (most recent epoch), or `final` (end of training).

In [ ]:
!python generate.py --project {PROJECT_NAME} --checkpoint best \
    --prompt "Once upon a time" \
    --max-new-tokens 200 \
    --temperature 0.8 \
    --top-k 50


## 8. (Optional) Keep training longer

Bump `EPOCHS` above and re-run the training cell (Section 5) — it will resume from
the last checkpoint rather than starting over. Increase `DATASET_PRESET` to `"medium"`
and re-run from Section 3 if you want to train on more data (tokenizer will need
retraining if the corpus changes a lot — delete `tokenizer.json` on Drive to force that).